# SGCRL Four Rooms — Absorbing Failure States Experiment
# Based on tabular_maze.ipynb, adds an absorbing region to test if the agent
# naturally learns to avoid it through contrastive learning alone.

In [ ]:
# Import required libraries
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 1, not GPU 0

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.animation as animation
from IPython.display import HTML, display

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

# Set up matplotlib for inline display
plt.ion()
%matplotlib inline

## Environment Setup


In [3]:
# Configure the Four Rooms Maze
HEIGHT, WIDTH = 10, 10
GOAL_COORD = (9, 9)  # row 9, col 9 (zero-indexed)
walls = np.zeros((HEIGHT, WIDTH), dtype=int)
DOOR_LEN = 2   

# Create horizontal wall
walls[HEIGHT // 2, :] = 1
doors_h = np.concatenate([
    WIDTH // 4 + np.arange(DOOR_LEN),
    WIDTH * 3 // 4 + np.arange(DOOR_LEN)
])
walls[HEIGHT // 2, doors_h] = 0

# Create vertical wall
walls[:, WIDTH // 2] = 1
doors_v = np.concatenate([
    HEIGHT // 4 + np.arange(DOOR_LEN),
    HEIGHT * 3 // 4 + np.arange(DOOR_LEN)
])
walls[doors_v, WIDTH // 2] = 0

# Environment parameters
NUM_STATES = HEIGHT * WIDTH
NUM_ACTIONS = 5  # stay, down, up, right, left
A_TO_DELTA = np.array([[0, 0],
                       [1, 0], [-1, 0],
                       [0, 1], [0, -1]])
START_STATE = np.ravel_multi_index((0, 0), walls.shape) 
GOAL_STATE = np.ravel_multi_index(GOAL_COORD, walls.shape)

print(f"Four Rooms Maze Environment Setup:")
print(f"   Grid size: {HEIGHT}x{WIDTH}")
print(f"   Number of states: {NUM_STATES}")
print(f"   Number of actions: {NUM_ACTIONS}")
print(f"   Start state: {START_STATE} -> {np.unravel_index(START_STATE, walls.shape)}")
print(f"   Goal state: {GOAL_STATE} -> {np.unravel_index(GOAL_STATE, walls.shape)}")

def step(state: int, action: int) -> int:
    """Deterministic step function for the maze."""
    di, dj = A_TO_DELTA[action]
    i, j   = np.unravel_index(state, walls.shape)
    ni, nj = i + di, j + dj
    if 0 <= ni < HEIGHT and 0 <= nj < WIDTH and walls[ni, nj] == 0:
        return np.ravel_multi_index((ni, nj), walls.shape)
    return state  # blocked: stay in place

def is_near_goal(s):
    """Check if state is near the goal (within Manhattan distance 1)"""
    si, sj = np.unravel_index(s, walls.shape)
    gi, gj = np.unravel_index(GOAL_STATE, walls.shape)
    return abs(si - gi) + abs(sj - gj) <= 1



Four Rooms Maze Environment Setup:
   Grid size: 10x10
   Number of states: 100
   Number of actions: 5
   Start state: 0 -> (0, 0)
   Goal state: 99 -> (9, 9)


## SGCRL Agent Implementation

In [ ]:
# JIT-compiled contrastive update (runs on GPU)
@jax.jit
def _jax_contrastive_update(psi, s_batch, sp_batch, lr):
    """Vectorised contrastive update on GPU via JAX."""
    psi_s = psi[s_batch]   # (B, D)
    psi_p = psi[sp_batch]  # (B, D)
    B = psi_s.shape[0]

    # Column-wise softmax probabilities
    dots = psi_s @ psi_p.T  # (B, B)
    dots = dots - dots.max(axis=0, keepdims=True)
    exp_logits = jnp.exp(dots)
    P = exp_logits / exp_logits.sum(axis=0, keepdims=True)

    diag_P = jnp.diag(P)
    nll = -jnp.mean(jnp.log(diag_P + 1e-12))

    # Anchor-state updates
    coeff = jnp.eye(B) - P
    anchor_update = lr * (coeff @ psi_p)  # (B, D)

    # Positive-state updates
    expected_anchor = P.T @ psi_s
    pos_update = lr * (psi_s - expected_anchor)  # (B, D)

    # Scatter-add updates into psi
    new_psi = psi.at[s_batch].add(anchor_update)
    new_psi = new_psi.at[sp_batch].add(pos_update)

    # Normalize
    norms = jnp.linalg.norm(new_psi, axis=1, keepdims=True) + 1e-8
    new_psi = new_psi / norms

    return new_psi, nll


class SGCRLAgent:
    """
    State-Goal Contrastive Reinforcement Learning Agent for Maze.
    Contrastive update runs on GPU via JAX. Episode collection on CPU.
    """
    
    def __init__(self, nState, nAction, rep_dim=16, episodes_per_upd=5, 
                 lr_psi=1e-3, replay_capacity=1000, max_steps=100, batch_size=128,
                 gamma=0.99, entropy_coeff=0.1, max_episodes=50000, plot_freq=100):
        # Store basic parameters
        self.nState = nState
        self.nAction = nAction
        self.rep_dim = rep_dim
        self.episodes_per_upd = episodes_per_upd
        self.lr_psi = lr_psi
        self.replay_capacity = replay_capacity
        self.max_steps = max_steps
        self.batch_size = batch_size
        self.gamma = gamma
        self.entropy_coeff = entropy_coeff
        self.max_episodes = max_episodes
        self.plot_freq = plot_freq
        self.norm = True
        
        # Set up goal
        self.goal = GOAL_STATE
        
        # Initialize representation
        self.psi = np.empty((nState, rep_dim))
        
        # Initialize goal embedding
        self.psi_goal = np.random.randn(rep_dim) * 0.1
        self.psi[self.goal] = self.psi_goal
        if self.norm:
            psi_norm = np.linalg.norm(self.psi[self.goal]) + 1e-8
            self.psi[self.goal] /= psi_norm
            self.psi_goal /= psi_norm
        
        # Initialize other states
        for s in range(nState):
            if s != self.goal:
                self.psi[s] = self.psi_goal + np.random.randn(rep_dim) * 0.1
        
        # Normalize psi to unit vectors
        if self.norm:
            psi_norms = np.linalg.norm(self.psi, axis=1, keepdims=True) + 1e-8
            self.psi /= psi_norms
        
        # JAX copy of psi (lives on GPU)
        self.psi_jax = jnp.array(self.psi)
        
        # Replay buffer and tracking
        self.replay = []
        self.visited_states = set()
        self.visited_counts = []
        self.loss_history = []
        self.success_list = []
        self.eval_success_list = []
        self.similarity_history = []

    def _sync_psi_to_numpy(self):
        """Copy JAX psi back to numpy for action selection."""
        self.psi = np.array(self.psi_jax)

    def select_action(self, s: int, g: int) -> int:
        """Softmax action selection (CPU, numpy)."""
        goal_vec = self.psi[g]
        similarities = []
        for a in range(self.nAction):
            ns = step(s, a)
            sim = self.psi[ns] @ goal_vec
            similarities.append(sim)
        logits = np.array(similarities) / self.entropy_coeff
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)
        return np.random.choice(self.nAction, p=probs)

    def eval_action(self, s: int, g: int) -> int:
        """Deterministic greedy action (CPU, numpy)."""
        goal_vec = self.psi[g]
        best_a, best_val = 0, -np.inf
        for a in range(self.nAction):
            ns = step(s, a)
            val = self.psi[ns] @ goal_vec
            if val > best_val:
                best_val = val
                best_a = a
        return best_a

    def collect_episode(self):
        """Generate one trajectory and push into replay buffer."""
        step_success = []
        traj = [START_STATE]
        for _ in range(self.max_steps):
            a = self.select_action(traj[-1], self.goal)
            ns = step(traj[-1], a)
            traj.append(ns)
            success = (ns == self.goal) or is_near_goal(ns)
            step_success.append(1 if success else 0)
        self.replay.append(traj)
        if len(self.replay) > self.replay_capacity:
            self.replay.pop(0)
        return step_success

    def run_eval_episode(self):
        """Generate one evaluation trajectory."""
        traj = [START_STATE]
        for _ in range(self.max_steps):
            a = self.eval_action(traj[-1], self.goal)
            ns = step(traj[-1], a)
            traj.append(ns)
        success = any((s == self.goal) or is_near_goal(s) for s in traj)
        return success, traj

    def update_representations(self) -> float:
        """Sample batch and run JIT'd contrastive update on GPU."""
        if len(self.replay) < 2:
            return 0.0

        traj_ids = np.random.choice(len(self.replay), self.batch_size, replace=True)
        s_list, sp_list = [], []
        for idx in traj_ids:
            traj = self.replay[idx]
            if len(traj) < 2:
                continue
            i = np.random.randint(0, len(traj) - 1)
            remaining = len(traj) - i
            w = self.gamma ** np.arange(remaining)
            w /= w.sum()
            j = i + np.random.choice(remaining, p=w)
            s_list.append(traj[i])
            sp_list.append(traj[j])

        if not s_list:
            return 0.0

        s_batch = jnp.array(s_list, dtype=jnp.int32)
        sp_batch = jnp.array(sp_list, dtype=jnp.int32)

        # GPU update
        self.psi_jax, nll = _jax_contrastive_update(
            self.psi_jax, s_batch, sp_batch, self.lr_psi)

        # Sync back to numpy for action selection
        self._sync_psi_to_numpy()

        return float(nll)
    
    def create_similarity_plot(self, episode, eval_traj):
        """Create similarity data for animation."""
        goal_vec = self.psi[self.goal]
        goal_norm = np.linalg.norm(goal_vec) + 1e-8
        sim = (self.psi @ goal_vec) / (np.linalg.norm(self.psi, axis=1) * goal_norm + 1e-8)
        self.similarity_history.append((episode, sim, eval_traj))

    def train(self):
        """Main training loop."""
        for ep in tqdm(range(self.max_episodes), desc="Training episodes"):
            ep_success = self.collect_episode()
            self.success_list.append(np.mean(ep_success))
            
            for state in self.replay[-1]:
                self.visited_states.add(state)
            self.visited_counts.append(len(self.visited_states))

            if ep % self.episodes_per_upd == 0:
                loss = self.update_representations()
                self.loss_history.append(loss)

            if (ep % self.plot_freq == 0 or ep == 0):
                eval_ep_success, eval_traj = self.run_eval_episode()
                self.eval_success_list.append(eval_ep_success)
                self.create_similarity_plot(ep, eval_traj)

    def plot_success_rate(self, window=100):
        """Plot success rate over training."""
        plt.figure(figsize=(8, 4))
        successes = np.array([1 if r > 0 else 0 for r in self.success_list])
        smoothed = []
        for i in range(len(successes)):
            start_idx = max(0, i - window + 1)
            smoothed.append(np.mean(successes[start_idx:i+1]))
        plt.plot(range(len(smoothed)), smoothed, color='tab:blue', linewidth=2)
        plt.xlabel('Training Episodes', fontsize=12)
        plt.ylabel('Success Rate', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        return None

## Configuration and Training

In [ ]:
config = {
    "rep_dim": 16,
    "episodes_per_upd": 1,
    "lr_psi": 1e-2,
    "replay_capacity": 50000,
    "max_steps": 500,
    "batch_size": 512,
    "max_episodes": 10000000,
    "gamma": 0.99,
    "entropy_coeff": 0.1,
    "plot_freq": 20000,
    "seed": 42
}

np.random.seed(config["seed"])

sgcrl_agent = SGCRLAgent(
    nState=NUM_STATES, 
    nAction=NUM_ACTIONS, 
    rep_dim=config["rep_dim"],
    episodes_per_upd=config["episodes_per_upd"],
    lr_psi=config["lr_psi"],
    replay_capacity=config["replay_capacity"],
    max_steps=config["max_steps"],
    batch_size=config["batch_size"],
    gamma=config["gamma"],
    entropy_coeff=config["entropy_coeff"],
    max_episodes=config["max_episodes"],
    plot_freq=config["plot_freq"]
)

# Random initialization (same as absorbing agent)
sgcrl_agent.psi = np.random.randn(NUM_STATES, config["rep_dim"]) * 0.1
psi_norms = np.linalg.norm(sgcrl_agent.psi, axis=1, keepdims=True) + 1e-8
sgcrl_agent.psi /= psi_norms
sgcrl_agent.psi_goal = sgcrl_agent.psi[GOAL_STATE].copy()
sgcrl_agent.psi_jax = jnp.array(sgcrl_agent.psi)  # sync to GPU

sgcrl_agent.train()
sgcrl_agent.plot_success_rate()
plt.title('Baseline Agent — Success Rate')
plt.show()

## Visualization of Training
### In the animation, we visualize the representational similarity of each state to the goal throughout training. We also show one trajecotry collected by the SGCRL policy with entropy regularization (actor). The success rate curves are computed using success of the actor trajectories.

In [15]:
def render_maze(ax, sim_map, traj, episode_num):
    """Render maze heatmap with complete trajectory"""
    ax.clear()
    
    # Remove border and ticks
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Background: similarity heat-map
    im = ax.imshow(sim_map, cmap="viridis", origin="lower", 
                  vmin=min(0.0, np.min(sim_map)), vmax=1)
    
    # Overlay walls
    ax.imshow(walls, cmap=plt.cm.binary, vmin=0, vmax=1,
             alpha=0.5, origin="lower")
    
    # Draw complete trajectory
    traj_coords = np.array([np.unravel_index(s, walls.shape) for s in traj])
    if len(traj_coords) > 1:
        ax.plot(traj_coords[:, 1], traj_coords[:, 0], color="black", linewidth=2.0, alpha=0.8)
    
    # Highlight start and goal
    ax.scatter(*np.unravel_index(START_STATE, walls.shape)[::-1],
              marker="o", s=200, c="lime", label="Start", zorder=10, 
              edgecolors='black', linewidth=2, clip_on=False)
    ax.scatter(*np.unravel_index(GOAL_STATE, walls.shape)[::-1],
              marker="*", s=300, c="red", label="Goal", zorder=10, 
              edgecolors='black', linewidth=2, clip_on=False)
    
    ax.set_title(f"Trial {episode_num}", fontsize=24)
    
    return im

def animate(ep_idx):
    """Animate by showing one frame per episode"""
    sim = similarities[ep_idx]
    sim_map = sim.reshape(walls.shape)
    traj = eval_trajs[ep_idx]
    episode_num = episodes[ep_idx]
    
    render_maze(ax, sim_map, traj, episode_num)
    
    return ax,

In [ ]:
# Extract data for animation
episodes = [item[0] for item in sgcrl_agent.similarity_history]
similarities = [item[1] for item in sgcrl_agent.similarity_history]
eval_trajs = [item[2] for item in sgcrl_agent.similarity_history]

# Create figure with single plot for maze visualization
fig, ax = plt.subplots(figsize=(10, 10))

print(f"Total frames to animate: {len(episodes)}")

# One frame per episode (no step-by-step animation)
anim = animation.FuncAnimation(fig, animate, frames=len(episodes), 
                                interval=200, blit=False, repeat=True)

# Increase the embed limit and optimize animation quality
plt.rcParams['animation.embed_limit'] = 100  

# Save animation as gif
anim.save('figures/maze_animation.gif', writer='pillow', fps=5)

# Display animation
display(HTML(anim.to_jshtml(fps=5, default_mode='loop')))
plt.close(fig)


Total frames to animate: 500


In [ ]:
# ── ABSORBING FAILURE STATES EXPERIMENT ────────────────────────────────
# We define an absorbing region (bottom-right quadrant) and retrain.
# The step function is modified so that entering the region freezes the state.
# The contrastive loss should naturally learn low similarity for absorbing states.
import matplotlib.patches as mpatches

In [ ]:
# Define absorbing region: bottom-right quadrant (rows 0-4, cols 5-9)
ABSORBING_REGION = set()
for r in range(0, 5):
    for c in range(5, 10):
        s = np.ravel_multi_index((r, c), walls.shape)
        if walls[r, c] == 0:
            ABSORBING_REGION.add(s)

# Save reference to original step BEFORE patching
_original_step = step

def step_absorbing(state: int, action: int) -> int:
    """Step function with absorbing states — once in region, state freezes."""
    if state in ABSORBING_REGION:
        return state  # absorbed: all actions loop back
    return _original_step(state, action)  # use saved reference, not global step

print(f"Absorbing region: {len(ABSORBING_REGION)} cells in bottom-right quadrant")

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))
display_grid = np.zeros((HEIGHT, WIDTH, 3))
for r in range(HEIGHT):
    for c in range(WIDTH):
        s = np.ravel_multi_index((r, c), walls.shape)
        if walls[r, c] == 1:
            display_grid[r, c] = [0.2, 0.2, 0.2]
        elif s in ABSORBING_REGION:
            display_grid[r, c] = [1.0, 0.7, 0.7]
        else:
            display_grid[r, c] = [1.0, 1.0, 1.0]
ax.imshow(display_grid, origin='lower')
ax.scatter(*GOAL_COORD[::-1], marker='*', s=300, c='red', zorder=5, label='Goal')
ax.scatter(0, 0, marker='o', s=200, c='lime', edgecolors='black', zorder=5, label='Start')
ax.set_title('Four Rooms — Absorbing Region (red shading)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train absorbing agent — same config, same random init
step = step_absorbing

np.random.seed(config["seed"])
agent_absorbing = SGCRLAgent(
    nState=NUM_STATES,
    nAction=NUM_ACTIONS,
    rep_dim=config["rep_dim"],
    episodes_per_upd=config["episodes_per_upd"],
    lr_psi=config["lr_psi"],
    replay_capacity=config["replay_capacity"],
    max_steps=config["max_steps"],
    batch_size=config["batch_size"],
    gamma=config["gamma"],
    entropy_coeff=config["entropy_coeff"],
    max_episodes=config["max_episodes"],
    plot_freq=config["plot_freq"]
)

# Random initialization (same as baseline)
agent_absorbing.psi = np.random.randn(NUM_STATES, config["rep_dim"]) * 0.1
psi_norms = np.linalg.norm(agent_absorbing.psi, axis=1, keepdims=True) + 1e-8
agent_absorbing.psi /= psi_norms
agent_absorbing.psi_goal = agent_absorbing.psi[GOAL_STATE].copy()

print("Training ABSORBING agent...")
agent_absorbing.train()

# Restore original step
step = _original_step

agent_absorbing.plot_success_rate()
plt.title('Absorbing Agent — Success Rate')
plt.show()

In [ ]:
# ── COMPARISON: Similarity maps ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, agent, title in [(ax1, sgcrl_agent, 'Baseline (no absorbing)'),
                          (ax2, agent_absorbing, 'Absorbing')]:
    goal_vec = agent.psi[agent.goal]
    goal_norm = np.linalg.norm(goal_vec) + 1e-8
    sim = (agent.psi @ goal_vec) / (np.linalg.norm(agent.psi, axis=1) * goal_norm + 1e-8)
    sim_map = sim.reshape(walls.shape)

    im = ax.imshow(sim_map, cmap='viridis', origin='lower',
                   vmin=min(0, sim.min()), vmax=1)
    ax.imshow(walls, cmap=plt.cm.binary, vmin=0, vmax=1, alpha=0.5, origin='lower')

    # Mark absorbing region boundary
    ax.add_patch(plt.Rectangle((4.5, -0.5), 5.5, 5.5,
        facecolor='none', edgecolor='red', linewidth=2, linestyle='--'))

    ax.scatter(*np.unravel_index(START_STATE, walls.shape)[::-1],
              marker='o', s=200, c='lime', edgecolors='black', linewidth=2, zorder=5)
    ax.scatter(*np.unravel_index(GOAL_STATE, walls.shape)[::-1],
              marker='*', s=300, c='red', edgecolors='black', linewidth=2, zorder=5)

    ax.set_title(f'{title} — ψ(s)·ψ(goal)', fontsize=14)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('figures/absorbing_similarity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print average similarity
for agent, name in [(sgcrl_agent, 'Baseline'), (agent_absorbing, 'Absorbing')]:
    goal_vec = agent.psi[agent.goal]
    goal_norm = np.linalg.norm(goal_vec) + 1e-8
    sim = (agent.psi @ goal_vec) / (np.linalg.norm(agent.psi, axis=1) * goal_norm + 1e-8)
    absorb_sims = [sim[s] for s in ABSORBING_REGION]
    non_absorb_sims = [sim[s] for s in range(NUM_STATES)
                       if s not in ABSORBING_REGION and walls.flat[s] == 0 and s != GOAL_STATE]
    print(f"{name}:  absorbing region avg sim = {np.mean(absorb_sims):.4f},  "
          f"non-absorbing avg sim = {np.mean(non_absorb_sims):.4f}")